# Лабораторная работа №1 — детекция объектов без YOLO

**Модель:** Faster R-CNN ResNet-50 FPN с COCO-весами.  
Ноутбук считывает изображение, преобразует его в RGB-тензор, запускает инференс и рисует детекции на исходной фотографии.

## 1. Подготовка окружения

Запустите ячейки сверху вниз. При первом запуске скачаются веса модели.

In [ ]:
!if [ -d /content/mii-lab1/.git ]; then cd /content/mii-lab1 && git pull --ff-only; else git clone -q https://github.com/A-m-u-r/mii-lab1.git /content/mii-lab1; fi
%cd /content/mii-lab1
!apt-get -qq update && apt-get -qq install -y fonts-dejavu-core
!pip -q install -r requirements-lab1.txt
!git rev-parse --short HEAD

## 2. Полный исходный код

Ниже выводится код, который используется для чтения изображения, препроцессинга, инференса и отрисовки.

In [ ]:
from IPython.display import Code, display

display(Code(filename='lab1_faster_rcnn.py', language='python'))

## 3. Загрузка исходного изображения

In [ ]:
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded))
print(f'Загружено: {image_path}')

## 4. Загрузка модели и препроцессинг

`preprocess_image` переводит изображение в RGB и float-тензор `[C, H, W]` в диапазоне `[0, 1]`.

In [ ]:
import importlib

import torch
from PIL import Image

import lab1_faster_rcnn
importlib.reload(lab1_faster_rcnn)
from lab1_faster_rcnn import (
    draw_detections,
    load_detector,
    run_inference,
)

font = lab1_faster_rcnn.load_label_font(20)
font_path = getattr(font, 'path', '')
assert 'DejaVuSans' in str(font_path), f'Не найден шрифт с кириллицей: {font_path}'
print(f'Шрифт подписей: {font_path}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
detector, coco_labels = load_detector(device)
print(f'Устройство: {device}')

## 5. Инференс и визуализация детекций

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

with Image.open(image_path) as opened_image:
    source_image = opened_image.convert('RGB')

detections = run_inference(
    model=detector,
    image=source_image,
    label_names=coco_labels,
    score_threshold=0.6,
    device=device,
)
annotated_image = draw_detections(source_image, detections)

output_path = Path('results/lab1_detections.jpg')
output_path.parent.mkdir(exist_ok=True)
annotated_image.save(output_path)

print(f'Найдено объектов: {len(detections)}')
for number, detection in enumerate(detections, start=1):
    print(f'{number}. {detection.label}: {detection.score:.2f}, box={detection.box}')

plt.figure(figsize=(14, 10))
plt.imshow(annotated_image)
plt.axis('off')
plt.title('Faster R-CNN: детекции')
plt.show()

## 6. Что сдавать

1. Выполните все ячейки (`Среда выполнения → Выполнить все`).
2. Убедитесь, что под последней ячейкой видны список детекций и итоговая картинка.
3. Сохраните копию в Google Drive или скачайте файл `.ipynb`.
4. Отправьте преподавателю ссылку на Colab/файл `.ipynb` и ссылку на GitHub-репозиторий.

**Вывод:** модель Faster R-CNN, не использующая YOLO, выполнила полный конвейер обработки изображения и визуализации детекций.